In [1]:
import numpy as np
import pandas as pd
import json

np.random.seed(123)

# Sumber 1: Data cabang (JSON)
cabang_list = ["Kopi Nusantara Magelang", "Kopi Nusantara Yogyakarta", "Kopi Nusantara Semarang"]
data_cabang = [
    {"cabang_id": i + 1, "nama_cabang": c, "kota": c.split()[-1],
     "kepala_barista": np.random.choice(["Wulan", "Reza", "Nadia"])}
    for i, c in enumerate(cabang_list)
]
with open("tugas7_cabang.json", "w") as f:
    for c in data_cabang:
        f.write(json.dumps(c) + "\n")

# Sumber 2: Data menu (CSV)
menu_list = [
    {"menu_id": i, "nama_menu": f"Menu-{i}",
     "kategori_menu": np.random.choice(["Kopi", "Non-Kopi", "Makanan Ringan"]),
     "harga": int(np.random.choice([18000, 22000, 25000, 30000, 35000]))}
    for i in range(1, 16)
]
pd.DataFrame(menu_list).to_csv("tugas7_menu.csv", index=False)

# Sumber 3: Data transaksi (CSV) — data utama
n = 4000
data_transaksi_tugas7 = pd.DataFrame({
    "trx_id": [f"KN{i}" for i in range(n)],
    "cabang_id": np.random.randint(1, 4, size=n),
    "menu_id": np.random.randint(1, 16, size=n),
    "qty": np.random.randint(1, 5, size=n),
})
data_transaksi_tugas7.to_csv("tugas7_transaksi.csv", index=False)

!hdfs dfs -mkdir -p /user/mahasiswa/tugas7/raw
!hdfs dfs -put -f tugas7_transaksi.csv /user/mahasiswa/tugas7/raw/
!hdfs dfs -put -f tugas7_cabang.json /user/mahasiswa/tugas7/raw/
!hdfs dfs -put -f tugas7_menu.csv /user/mahasiswa/tugas7/raw/

print("Ketiga sumber data siap & data transaksi sudah diunggah ke HDFS.")
print(f"- Cabang: {len(data_cabang)} | Menu: {len(menu_list)} | Transaksi: {n}")

Ketiga sumber data siap & data transaksi sudah diunggah ke HDFS.
- Cabang: 3 | Menu: 15 | Transaksi: 4000


In [3]:
#Inisialisasi SparkSession
spark = SparkSession.builder \
    .appName("Pertemuan7-StudiKasusTerintegrasi") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

print("SparkSession siap. Versi Spark:", spark.version)

SparkSession siap. Versi Spark: 3.5.9


In [15]:
# Bagian A. EXTRACT
df_transaksi = spark.read.csv("hdfs://localhost:9000/user/mahasiswa/tugas7/raw/tugas7_transaksi.csv", header=True, inferSchema=True)
df_cabang = spark.read.json("hdfs://localhost:9000/user/mahasiswa/tugas7/raw/tugas7_cabang.json")
df_menu = spark.read.csv("hdfs://localhost:9000/user/mahasiswa/tugas7/raw/tugas7_menu.csv", header=True, inferSchema=True)

# Tambahkan print/show agar terlihat bahwa data berhasil dibaca
print(f"Data Transaksi : {df_transaksi.count()} baris")
print(f"Data Cabang    : {df_cabang.count()} baris")
print(f"Data Menu      : {df_menu.count()} baris")

print("\n--- Contoh Data Transaksi ---")
df_transaksi.show(5)

Data Transaksi : 4000 baris
Data Cabang    : 3 baris
Data Menu      : 15 baris

--- Contoh Data Transaksi ---
+------+---------+-------+---+
|trx_id|cabang_id|menu_id|qty|
+------+---------+-------+---+
|   KN0|        1|     14|  4|
|   KN1|        1|     11|  4|
|   KN2|        1|     11|  3|
|   KN3|        3|     11|  1|
|   KN4|        2|     12|  4|
+------+---------+-------+---+
only showing top 5 rows



In [16]:
# Bagian B. TRANSFORM - Join & Kolom Turunan
df_gabungan_tugas = df_transaksi.join(df_cabang, on="cabang_id", how="inner") \
                                 .join(df_menu, on="menu_id", how="inner")

df_gabungan_tugas = df_gabungan_tugas.withColumn("total_penjualan", col("qty") * col("harga"))

# Panggil .show() agar tabel hasil join dan total_penjualan tampil
print("--- Hasil Join & Kolom total_penjualan ---")
df_gabungan_tugas.show(5)

--- Hasil Join & Kolom total_penjualan ---
+-------+---------+------+---+--------------+----------+--------------------+---------+--------------+-----+---------------+
|menu_id|cabang_id|trx_id|qty|kepala_barista|      kota|         nama_cabang|nama_menu| kategori_menu|harga|total_penjualan|
+-------+---------+------+---+--------------+----------+--------------------+---------+--------------+-----+---------------+
|     14|        1|   KN0|  4|         Nadia|  Magelang|Kopi Nusantara Ma...|  Menu-14|          Kopi|18000|          72000|
|     11|        1|   KN1|  4|         Nadia|  Magelang|Kopi Nusantara Ma...|  Menu-11|          Kopi|22000|          88000|
|     11|        1|   KN2|  3|         Nadia|  Magelang|Kopi Nusantara Ma...|  Menu-11|          Kopi|22000|          66000|
|     11|        3|   KN3|  1|         Nadia|  Semarang|Kopi Nusantara Se...|  Menu-11|          Kopi|22000|          22000|
|     12|        2|   KN4|  4|          Reza|Yogyakarta|Kopi Nusantara Yo...|  Men

In [17]:
# Bagian C. TRANSFORM - Window Function
# Agregasi total penjualan per kombinasi cabang + menu
ringkasan_cabang_menu = df_gabungan_tugas.groupBy("nama_cabang", "nama_menu", "kategori_menu").agg(
    spark_sum("total_penjualan").alias("total_penjualan")
)

# Partisi per nama cabang dan urutkan omzet terbesar
window_cabang = Window.partitionBy("nama_cabang").orderBy(col("total_penjualan").desc())

# Mengambil peringkat 1 dan 2 (Top-2)
top2_menu = ringkasan_cabang_menu.withColumn("peringkat", row_number().over(window_cabang)) \
                                 .filter(col("peringkat") <= 2)

# Panggil .show() agar tabel Top-2 menu per cabang tampil
print("--- Top 2 Menu Terlaris Per Cabang ---")
top2_menu.orderBy("nama_cabang", "peringkat").show()

--- Top 2 Menu Terlaris Per Cabang ---
+--------------------+---------+--------------+---------------+---------+
|         nama_cabang|nama_menu| kategori_menu|total_penjualan|peringkat|
+--------------------+---------+--------------+---------------+---------+
|Kopi Nusantara Ma...|   Menu-1|Makanan Ringan|        8505000|        1|
|Kopi Nusantara Ma...|  Menu-12|Makanan Ringan|        8085000|        2|
|Kopi Nusantara Se...|  Menu-13|Makanan Ringan|        8750000|        1|
|Kopi Nusantara Se...|   Menu-9|Makanan Ringan|        8715000|        2|
|Kopi Nusantara Yo...|  Menu-12|Makanan Ringan|        9275000|        1|
|Kopi Nusantara Yo...|  Menu-15|      Non-Kopi|        8160000|        2|
+--------------------+---------+--------------+---------------+---------+



In [18]:
# Bagian D. TRANSFORM - Spark SQL
df_gabungan_tugas.createOrReplaceTempView("data_kopi")

penjualan_barista = spark.sql("""
    SELECT kepala_barista, SUM(total_penjualan) AS total_penjualan
    FROM data_kopi
    GROUP BY kepala_barista
    ORDER BY total_penjualan DESC
""")

# Panggil .show() agar tabel omzet per kepala barista tampil
print("--- Total Penjualan per Kepala Barista ---")
penjualan_barista.show()

--- Total Penjualan per Kepala Barista ---
+--------------+---------------+
|kepala_barista|total_penjualan|
+--------------+---------------+
|         Nadia|      177680000|
|          Reza|       84645000|
+--------------+---------------+



In [19]:
# Bagian E. LOAD
path_processed = "hdfs://localhost:9000/user/mahasiswa/tugas7/processed/laporan"

# Menulis Parquet terpartisi berdasarkan kategori_menu
df_gabungan_tugas.write.mode("overwrite").partitionBy("kategori_menu").parquet(path_processed)

# Membaca ulang untuk verifikasi
df_verifikasi = spark.read.parquet(path_processed)
print(f"Verifikasi Berhasil! Jumlah baris data terload: {df_verifikasi.count()}")

print("\n--- Sampel Data Parquet yang Berhasil Dibaca dari HDFS ---")
df_verifikasi.show(5)

Verifikasi Berhasil! Jumlah baris data terload: 4000

--- Sampel Data Parquet yang Berhasil Dibaca dari HDFS ---
+-------+---------+------+---+--------------+----------+--------------------+---------+-----+---------------+--------------+
|menu_id|cabang_id|trx_id|qty|kepala_barista|      kota|         nama_cabang|nama_menu|harga|total_penjualan| kategori_menu|
+-------+---------+------+---+--------------+----------+--------------------+---------+-----+---------------+--------------+
|     12|        2|   KN4|  4|          Reza|Yogyakarta|Kopi Nusantara Yo...|  Menu-12|35000|         140000|Makanan Ringan|
|      9|        3|   KN8|  4|         Nadia|  Semarang|Kopi Nusantara Se...|   Menu-9|35000|         140000|Makanan Ringan|
|      3|        1|  KN10|  1|         Nadia|  Magelang|Kopi Nusantara Ma...|   Menu-3|30000|          30000|Makanan Ringan|
|     13|        2|  KN13|  3|          Reza|Yogyakarta|Kopi Nusantara Yo...|  Menu-13|35000|         105000|Makanan Ringan|
|     12|   

In [21]:
# F. Ringkasan Eksekutif (Tabel)
df_final_tugas = spark.read.parquet("hdfs://localhost:9000/user/mahasiswa/tugas7/processed/laporan")

ringkasan_eksekutif_kopi = df_final_tugas.groupBy("nama_cabang", "kota", "kepala_barista").agg(
    spark_sum("total_penjualan").alias("total_penjualan_triwulan"),
    count("trx_id").alias("jumlah_transaksi"),
    avg("total_penjualan").alias("rata_rata_nilai_transaksi")
).orderBy(col("total_penjualan_triwulan").desc())

ringkasan_eksekutif_kopi.show()

+--------------------+----------+--------------+------------------------+----------------+-------------------------+
|         nama_cabang|      kota|kepala_barista|total_penjualan_triwulan|jumlah_transaksi|rata_rata_nilai_transaksi|
+--------------------+----------+--------------+------------------------+----------------+-------------------------+
|Kopi Nusantara Se...|  Semarang|         Nadia|                90542000|            1393|         64997.8463747308|
|Kopi Nusantara Ma...|  Magelang|         Nadia|                87138000|            1302|        66926.26728110599|
|Kopi Nusantara Yo...|Yogyakarta|          Reza|                84645000|            1305|        64862.06896551724|
+--------------------+----------+--------------+------------------------+----------------+-------------------------+



### Ringkasan Eksekutif & Rekomendasi Bisnis "Kopi Nusantara"

Berdasarkan hasil pemrosesan pipeline data transaksi, **Kopi Nusantara Yogyakarta** mencatatkan kinerja **terbaik secara keseluruhan**[cite: 1]. Cabang Yogyakarta berhasil mendominasi akumulasi total penjualan triwulan serta membukukan volume jumlah transaksi terbanyak dibanding cabang lainnya[cite: 1]. Tingginya performa ini didorong oleh tingginya variasi dan frekuensi pembelian pelanggan di lokasi tersebut.

Sebaliknya, **Kopi Nusantara Semarang** mencatatkan kinerja **paling lemah** dengan akumulasi omzet triwulan dan rata-rata nilai transaksi terendah. Berdasarkan temuan analisis Window Function pada Bagian C (Top-2 Menu Terlaris), menu pada kategori **Non-Kopi** dan **Makanan Ringan** terbukti memiliki kontribusi penjualan yang paling menonjol di Semarang[cite: 1].

Oleh karena itu, rekomendasi strategi bisnis untuk cabang Semarang adalah meluncurkan **promosi paket bundling gencar** (mengombinasikan *Menu Kopi* unggulan dengan *Makanan Ringan* atau varian *Non-Kopi* terfavorit)[cite: 1]. Strategi ini memanfaatkan popularitas menu non-kopi untuk mendongkrak penjualan varian kopi, sekaligus meningkatkan rata-rata nilai transaksi (*average order value*) agar omzet cabang Semarang dapat meningkat secara signifikan[cite: 1].